# Build `comparisons_df.pickle` from `merged_scores.csv`

This notebook:

1. Loads the raw merged survey CSV (**no header**).
2. Parses image URLs into `(dataset, image_filename)` pairs.
3. Produces a dataframe compatible with your `train.py` + `data.py` pipeline.
4. Saves:
   - `comparisons_df.pickle` (primary artifact for training)
   - `comparisons_df.parquet` (optional, for fast inspection)
5. Prints dataset/class statistics.

**Column convention in `merged_scores.csv` (no header):**
- col 0: timestamp (string)
- col 1: user identifier (string)
- col 2: left image URL (string)
- col 3: right image URL (string)
- col 4: score in {-1, 0, +1} (int)


In [1]:
# =========================================
# 1) Imports & configuration
# =========================================
from pathlib import Path
import re
import pandas as pd
import numpy as np

# Path to the merged raw scores you generated earlier
MERGED_SCORES_CSV = Path("cities_dataset/merged_scores.csv")  # assumes notebook is in same folder as merged_scores.csv

# Where to save the training dataframe
OUT_PICKLE = Path("comparisons_without_eyetracking_df.pickle")

# Optional extra outputs (handy for quick inspection)
OUT_PARQUET = Path("comparisons_df.parquet")

print("Input :", MERGED_SCORES_CSV.resolve())
print("Output:", OUT_PICKLE.resolve())


Input : /home/csantiago/cities_dataset/merged_scores.csv
Output: /home/csantiago/comparisons_without_eyetracking_df.pickle


In [2]:
# =========================================
# 2) Load merged_scores.csv (NO HEADER)
# =========================================
# Important: merged_scores.csv has NO header row. The first row is real data.
raw = pd.read_csv(MERGED_SCORES_CSV, header=None)
raw.columns = ["timestamp", "user_id", "url_l", "url_r", "score_raw"]

# Basic sanity
print("Rows:", len(raw))
print(raw.head(3))
print("\nScore distribution (raw):")
print(raw["score_raw"].value_counts().sort_index())


Rows: 13178
             timestamp                                            user_id  \
0  2022-09-26 08:40:54                                         migueltest   
1  2022-09-26 13:35:29   cyclingee65858ab02db80be21b910b76652c086fae44...   
2  2022-09-26 13:35:46   cyclingee65858ab02db80be21b910b76652c086fae44...   

                                               url_l  \
0   http://smartbike.isr.tecnico.ulisboa.pt/subje...   
1   http://smartbike.isr.tecnico.ulisboa.pt/subje...   
2   http://smartbike.isr.tecnico.ulisboa.pt/subje...   

                                               url_r  score_raw  
0   http://smartbike.isr.tecnico.ulisboa.pt/subje...          1  
1   http://smartbike.isr.tecnico.ulisboa.pt/subje...          1  
2   http://smartbike.isr.tecnico.ulisboa.pt/subje...          0  

Score distribution (raw):
score_raw
-1    4327
 0    4144
 1    4707
Name: count, dtype: int64


In [3]:
# =========================================
# 3) Parse URLs -> (dataset, filename)
# =========================================
# Your training pipeline expects:
#   - 'dataset' : subfolder under images/ (e.g., berlin, barcelona, sequences, ...)
#   - 'image_l' : filename (e.g., 123.jpg, si_3_4.jpg)
#   - 'image_r' : filename
#
# Many URLs have the pattern:
#   .../subjectivesafety_images/<dataset>/<filename>.jpg
# Example:
#   http://.../subjectivesafety_images/sequences/fi_0_1.jpg
#
# Some older/test URLs may be:
#   .../subjectivesafety_images/4.jpg
# In that case, dataset is unknown; we store dataset='__root__' so train.py will look in images/__root__/4.jpg.
# You can rename/move this folder later if needed.

def parse_subjectivesafety_url(url: str):
    """Return (dataset, filename) from a Smartbike subjectivesafety_images URL."""
    if pd.isna(url):
        return None, None

    url = str(url).strip()
    # Normalize accidental whitespace
    url = re.sub(r"\s+", "", url)

    # Capture the path after 'subjectivesafety_images/'
    m = re.search(r"subjectivesafety_images/(.+)$", url, flags=re.IGNORECASE)
    if not m:
        # Not matching expected host/path; fallback to filename only
        return "__unknown__", Path(url).name

    tail = m.group(1)  # e.g. 'sequences/fi_0_1.jpg' OR '4.jpg'
    parts = tail.split("/")

    if len(parts) == 1:
        dataset = "__root__"
        filename = parts[0]
    else:
        dataset = parts[0]
        filename = parts[-1]

    return dataset, filename


parsed_l = raw["url_l"].apply(parse_subjectivesafety_url)
parsed_r = raw["url_r"].apply(parse_subjectivesafety_url)

raw[["dataset_l", "image_l"]] = pd.DataFrame(parsed_l.tolist(), index=raw.index)
raw[["dataset_r", "image_r"]] = pd.DataFrame(parsed_r.tolist(), index=raw.index)

# If left and right disagree, we keep dataset_l but record a flag for debugging.
raw["dataset_mismatch"] = raw["dataset_l"] != raw["dataset_r"]

print("Dataset mismatches (L vs R):", int(raw["dataset_mismatch"].sum()))
display(raw.loc[raw["dataset_mismatch"], ["url_l","url_r","dataset_l","dataset_r"]].head(5))


Dataset mismatches (L vs R): 0


,url_l,url_r,dataset_l,dataset_r


In [4]:
# =========================================
# 4) Build training dataframe (train.py/data.py compatible)
# =========================================
# Your train.py read_data() selects these columns if present:
#   'score', 'image_l', 'image_r', 'dataset', 'has_eyetracker',
#   'npy_file_l', 'npy_file_r', 'survey_id', 'trial_id'
#
# We'll build them with conservative defaults:
#   - has_eyetracker: False  (because merged_scores.csv is generic survey data)
#   - npy_file_l/r : None
#   - survey_id   : user_id  (so you can group by respondent later)
#   - trial_id    : incremental counter per user_id
#
# Score must be int in {-1, 0, +1}.

df = pd.DataFrame({
    "score": pd.to_numeric(raw["score_raw"], errors="coerce"),
    "dataset": raw["dataset_l"].astype(str),
    "image_l": raw["image_l"].astype(str),
    "image_r": raw["image_r"].astype(str),
    "has_eyetracker": False,
    "npy_file_l": None,
    "npy_file_r": None,
    "survey_id": raw["user_id"].astype(str),
})

# Drop rows with invalid scores or missing filenames
before = len(df)
df = df[df["score"].isin([-1, 0, 1])].copy()
df = df[(df["image_l"].str.len() > 0) & (df["image_r"].str.len() > 0)].copy()

# Ensure int dtype for score
df["score"] = df["score"].astype(int)

# Build trial_id per user (helps debugging and reproducibility)
df["trial_id"] = df.groupby("survey_id").cumcount().astype(int)

print(f"Dropped {before-len(df)} invalid rows; kept {len(df)} rows.")
print(df.head(3))


Dropped 0 invalid rows; kept 13178 rows.
   score    dataset     image_l     image_r  has_eyetracker npy_file_l  \
0      1  sequences  fi_0_1.jpg  fi_0_2.jpg           False       None   
1      1     berlin       9.jpg    2106.jpg           False       None   
2      0  sequences  li_4_5.jpg  li_4_6.jpg           False       None   

  npy_file_r                                          survey_id  trial_id  
0       None                                         migueltest         0  
1       None   cyclingee65858ab02db80be21b910b76652c086fae44...         0  
2       None   cyclingee65858ab02db80be21b910b76652c086fae44...         1  


In [5]:
# =========================================
# 5) Save artifacts for training
# =========================================
# This is the file your train.py expects (via --comparisons comparisons_df.pickle)
df.to_pickle(OUT_PICKLE)
print("Wrote:", OUT_PICKLE.resolve())

# Optional: Parquet for easy/fast inspection
try:
    df.to_parquet(OUT_PARQUET, index=False)
    print("Wrote:", OUT_PARQUET.resolve())
except Exception as e:
    print("[INFO] Parquet write skipped (missing engine like pyarrow/fastparquet):", e)


Wrote: /home/csantiago/comparisons_without_eyetracking_df.pickle
[INFO] Parquet write skipped (missing engine like pyarrow/fastparquet): Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.


In [6]:
# =========================================
# 6) Final statistics (useful for thesis + sanity)
# =========================================
import numpy as np

def pct(x):
    return 100.0 * x / max(1, x.sum())

print("\n==================== DATASET OVERVIEW ====================")
print("Total comparisons:", len(df))
print("Unique users      :", df["survey_id"].nunique())
print("Datasets          :", df["dataset"].nunique())
print("Dataset list      :", sorted(df["dataset"].unique())[:20], "..." if df["dataset"].nunique() > 20 else "")

# 6.1) Class distribution per dataset
print("\n-------------------- Class distribution per dataset --------------------")
rows = []
for dset, g in df.groupby("dataset"):
    counts = g["score"].value_counts().reindex([-1, 0, 1], fill_value=0)
    rows.append({
        "dataset": dset,
        "n": len(g),
        "pct_-1": 100.0 * counts[-1] / len(g),
        "pct_0":  100.0 * counts[0]  / len(g),
        "pct_+1": 100.0 * counts[1]  / len(g),
    })

stats_classes = pd.DataFrame(rows).sort_values("n", ascending=False)
display(stats_classes.head(30))

# 6.2) Average appearance of images per dataset
# We count how many times each image appears as left OR right within a dataset.
print("\n-------------------- Average image appearances per dataset --------------------")
rows = []
for dset, g in df.groupby("dataset"):
    # count appearances across both columns
    counts_l = g["image_l"].value_counts()
    counts_r = g["image_r"].value_counts()
    counts = counts_l.add(counts_r, fill_value=0)

    rows.append({
        "dataset": dset,
        "n_comparisons": len(g),
        "n_unique_images": int(counts.shape[0]),
        "avg_appearances_per_image": float(counts.mean()),
        "median_appearances_per_image": float(counts.median()),
        "max_appearances_single_image": int(counts.max()),
        "min_appearances_single_image": int(counts.min()),
    })

stats_images = pd.DataFrame(rows).sort_values("n_comparisons", ascending=False)
display(stats_images.head(30))

# 6.3) Additional helpful stats
print("\n-------------------- Additional helpful stats --------------------")

# How many rows have a dataset mismatch in the original parsing
# (should be ~0; if not, inspect the raw URLs)
mismatch_rate = (raw["dataset_mismatch"].mean() * 100.0) if "dataset_mismatch" in raw.columns else 0.0
print(f"Dataset mismatch rate (L vs R): {mismatch_rate:.4f}%")

# Overall class balance
overall_counts = df["score"].value_counts().reindex([-1,0,1], fill_value=0)
print("Overall class distribution:")
print(overall_counts.to_frame("count").assign(pct=(overall_counts / len(df) * 100.0)))

# Comparisons per user (distribution)
per_user = df.groupby("survey_id").size()
print("\nComparisons per user:")
print(per_user.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

print("\nDone.")



==================== DATASET OVERVIEW ====================
Total comparisons: 13178
Unique users      : 262
Datasets          : 8
Dataset list      : ['__root__', 'barcelona', 'berlin', 'london_uk_collideoscope', 'london_uk_gov', 'munich', 'paris', 'sequences'] 

-------------------- Class distribution per dataset --------------------


,dataset,n,pct_-1,pct_0,pct_+1
2,berlin,6550,38.610687,21.541985,39.847328
7,sequences,2987,17.040509,58.185470,24.774021
1,barcelona,1198,33.889816,28.380634,37.729549
3,london_uk_collideoscope,570,36.842105,30.175439,32.982456
4,london_uk_gov,569,33.391916,32.337434,34.270650
6,paris,556,32.374101,32.194245,35.431655
5,munich,542,37.453875,19.926199,42.619926
0,__root__,206,48.543689,5.825243,45.631068



-------------------- Average image appearances per dataset --------------------


,dataset,n_comparisons,n_unique_images,avg_appearances_per_image,median_appearances_per_image,max_appearances_single_image,min_appearances_single_image
2,berlin,6550,4456,2.939856,3.0,19,1
7,sequences,2987,378,15.804233,11.0,41,6
1,barcelona,1198,1469,1.631042,1.0,7,1
3,london_uk_collideoscope,570,992,1.149194,1.0,5,1
4,london_uk_gov,569,970,1.173196,1.0,4,1
6,paris,556,584,1.904110,2.0,8,1
5,munich,542,918,1.180828,1.0,5,1
0,__root__,206,10,41.200000,41.5,51,29



-------------------- Additional helpful stats --------------------
Dataset mismatch rate (L vs R): 0.0000%
Overall class distribution:
       count        pct
score                  
-1      4327  32.835028
 0      4144  31.446350
 1      4707  35.718622

Comparisons per user:
count    262.000000
mean      50.297710
std       36.776366
min        1.000000
10%        4.000000
25%       15.250000
50%       65.000000
75%       65.000000
90%       70.000000
max      333.000000
dtype: float64

Done.


## 7) Merge with eyetracker comparisons (optional but recommended)

If you have `eyetracker_comparisons_df.pickle` (pairs that include gaze/heatmaps), this cell merges it with the survey-only dataframe built above.

Output:
- `comparisons_df.pickle` (recommended final file to use in `train.py` via `--comparisons`)


In [7]:
# =========================================
# 7) Merge survey comparisons with eyetracker comparisons
# =========================================
from pathlib import Path
import pandas as pd
import numpy as np

# Path to your eyetracker dataframe (update if needed)
EYE_PICKLE = Path("eyetracker_comparisons_df.pickle")

# Final combined output
OUT_PICKLE_ALL = Path("comparisons_df.pickle")

# -----------------------------
# 7.1) Load eyetracker df
# -----------------------------
# Accept both pickle.load and pd.read_pickle formats transparently.
try:
    et_df = pd.read_pickle(EYE_PICKLE)
except Exception:
    import pickle
    et_df = pickle.load(open(EYE_PICKLE, "rb"))

print("Eyetracker rows:", len(et_df))
print("Eyetracker columns:", list(et_df.columns))

# -----------------------------
# 7.2) Normalize eyetracker columns to match train.py expectations
# -----------------------------
# train.py read_data() expects, if present:
#   ['score','image_l','image_r','dataset','has_eyetracker','npy_file_l','npy_file_r','survey_id','trial_id']
#
# Your eyetracker df typically already includes most of these.
# We defensively ensure required columns exist and have correct types.

# 7.2.a) Ensure core columns exist
required = ["score", "image_l", "image_r"]
missing = [c for c in required if c not in et_df.columns]
if missing:
    raise ValueError(f"Eyetracker df missing required columns: {missing}")

# 7.2.b) dataset column (fallback if absent)
if "dataset" not in et_df.columns:
    et_df["dataset"] = "__unknown__"

# 7.2.c) has_eyetracker flag must be True for these rows
et_df["has_eyetracker"] = True

# 7.2.d) gaze map pointers (if missing, create as None; your data.py can handle absent gaze if has_eyetracker False)
for col in ["npy_file_l", "npy_file_r"]:
    if col not in et_df.columns:
        et_df[col] = None

# 7.2.e) survey_id / trial_id (if absent, set safe defaults)
if "survey_id" not in et_df.columns:
    et_df["survey_id"] = "eyetracker_unknown"
if "trial_id" not in et_df.columns:
    et_df["trial_id"] = et_df.groupby("survey_id").cumcount().astype(int)

# 7.2.f) Ensure jpg suffix for filenames (your train.py also does this, but keeping it consistent is helpful)
def _ensure_jpg(x: str) -> str:
    x = str(x)
    return x if x.lower().endswith(".jpg") else f"{x}.jpg"

et_df["image_l"] = et_df["image_l"].astype(str).apply(_ensure_jpg)
et_df["image_r"] = et_df["image_r"].astype(str).apply(_ensure_jpg)

# 7.2.g) Ensure score is int in {-1,0,1}
et_df["score"] = pd.to_numeric(et_df["score"], errors="coerce")
et_df = et_df[et_df["score"].isin([-1, 0, 1])].copy()
et_df["score"] = et_df["score"].astype(int)

# -----------------------------
# 7.3) Align schemas & concatenate
# -----------------------------
# df is the survey dataframe built previously (Cell 4).
# We will align both dataframes to the union of columns so nothing is lost.

# Ensure survey df has the same gaze pointer columns (as None)
for col in ["npy_file_l", "npy_file_r"]:
    if col not in df.columns:
        df[col] = None

# Ensure survey df has dataset, survey_id, trial_id, has_eyetracker
for col, default in [
    ("dataset", "__unknown__"),
    ("survey_id", "survey_unknown"),
    ("trial_id", None),
    ("has_eyetracker", False),
]:
    if col not in df.columns:
        df[col] = default

# If trial_id missing on survey side (should not happen if you used the notebook), rebuild it safely
if df["trial_id"].isna().any():
    df["trial_id"] = df.groupby("survey_id").cumcount().astype(int)

# Concatenate
df_all = pd.concat([df, et_df], ignore_index=True, sort=False)

print("\nCombined rows:", len(df_all))
print(" - survey-only rows   :", int((df_all["has_eyetracker"] == False).sum()))
print(" - eyetracker rows    :", int((df_all["has_eyetracker"] == True).sum()))

# -----------------------------
# 7.4) Save final pickle
# -----------------------------
df_all.to_pickle(OUT_PICKLE_ALL)
print("Wrote:", OUT_PICKLE_ALL.resolve())

# =========================================
# 7.5) Extended combined sanity statistics
# =========================================
import numpy as np
import pandas as pd

print("\n==================== COMBINED DATASET OVERVIEW ====================")
print("Total comparisons        :", len(df_all))
print("Unique users             :", df_all["survey_id"].nunique())
print("Datasets                 :", df_all["dataset"].nunique())
print("With eyetracker          :", int(df_all["has_eyetracker"].sum()))
print("Without eyetracker       :", int((~df_all["has_eyetracker"]).sum()))

# -------------------------------------------------
# 7.5.1) Overall class distribution
# -------------------------------------------------
print("\n-------------------- Overall class distribution --------------------")
overall = df_all["score"].value_counts().reindex([-1, 0, 1], fill_value=0)
overall_df = pd.DataFrame({
    "count": overall,
    "pct": overall / len(df_all) * 100.0
})
display(overall_df)

# -------------------------------------------------
# 7.5.2) Class distribution per dataset (percent)
# -------------------------------------------------
print("\n-------------------- Class distribution per dataset --------------------")
rows = []
for dset, g in df_all.groupby("dataset"):
    counts = g["score"].value_counts().reindex([-1, 0, 1], fill_value=0)
    rows.append({
        "dataset": dset,
        "n": len(g),
        "pct_-1": 100.0 * counts[-1] / len(g),
        "pct_0":  100.0 * counts[0]  / len(g),
        "pct_+1": 100.0 * counts[1]  / len(g),
    })

stats_classes = (
    pd.DataFrame(rows)
    .sort_values("n", ascending=False)
)

display(stats_classes.head(30))

# -------------------------------------------------
# 7.5.3) Class distribution per dataset × eyetracker
# -------------------------------------------------
print("\n-------------------- Class distribution per dataset × eyetracker --------------------")
rows = []
for (dset, has_et), g in df_all.groupby(["dataset", "has_eyetracker"]):
    counts = g["score"].value_counts().reindex([-1, 0, 1], fill_value=0)
    rows.append({
        "dataset": dset,
        "has_eyetracker": bool(has_et),
        "n": len(g),
        "pct_-1": 100.0 * counts[-1] / len(g),
        "pct_0":  100.0 * counts[0]  / len(g),
        "pct_+1": 100.0 * counts[1]  / len(g),
    })

stats_classes_et = (
    pd.DataFrame(rows)
    .sort_values(["dataset", "has_eyetracker"], ascending=[True, False])
)

display(stats_classes_et.head(40))

# -------------------------------------------------
# 7.5.4) Image appearance statistics per dataset
# -------------------------------------------------
print("\n-------------------- Image appearance stats per dataset --------------------")
rows = []
for dset, g in df_all.groupby("dataset"):
    counts_l = g["image_l"].value_counts()
    counts_r = g["image_r"].value_counts()
    counts = counts_l.add(counts_r, fill_value=0)

    rows.append({
        "dataset": dset,
        "n_comparisons": len(g),
        "n_unique_images": int(counts.shape[0]),
        "avg_appearances_per_image": float(counts.mean()),
        "median_appearances_per_image": float(counts.median()),
        "max_appearances_single_image": int(counts.max()),
        "min_appearances_single_image": int(counts.min()),
    })

stats_images = (
    pd.DataFrame(rows)
    .sort_values("n_comparisons", ascending=False)
)

display(stats_images.head(30))

# -------------------------------------------------
# 7.5.5) Eyetracker coverage per dataset
# -------------------------------------------------
print("\n-------------------- Eyetracker coverage per dataset --------------------")
et_cov = (
    df_all.groupby("dataset")["has_eyetracker"]
    .agg(
        n_total="count",
        n_with_et="sum",
        pct_with_et=lambda x: 100.0 * x.mean()
    )
    .sort_values("n_total", ascending=False)
)

display(et_cov.head(30))

# -------------------------------------------------
# 7.5.6) Comparisons per user (global)
# -------------------------------------------------
print("\n-------------------- Comparisons per user --------------------")
per_user = df_all.groupby("survey_id").size()
display(per_user.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

print("\nDone.")



Eyetracker rows: 1495
Eyetracker columns: ['dataset', 'image_l', 'image_r', 'score', 'has_eyetracker', 'survey_id', 'trial_id', 'npy_file_l', 'npy_file_r']

Combined rows: 14673
 - survey-only rows   : 13178
 - eyetracker rows    : 1495
Wrote: /home/csantiago/comparisons_df.pickle

==================== COMBINED DATASET OVERVIEW ====================
Total comparisons        : 14673
Unique users             : 285
Datasets                 : 8
With eyetracker          : 1495
Without eyetracker       : 13178

-------------------- Overall class distribution --------------------


,count,pct
score,,
-1,5032,34.294282
0,4144,28.242350
1,5497,37.463368



-------------------- Class distribution per dataset --------------------


,dataset,n,pct_-1,pct_0,pct_+1
2,berlin,7549,40.058286,18.691217,41.250497
7,sequences,3483,20.643124,49.899512,29.457364
1,barcelona,1198,33.889816,28.380634,37.729549
3,london_uk_collideoscope,570,36.842105,30.175439,32.982456
4,london_uk_gov,569,33.391916,32.337434,34.270650
6,paris,556,32.374101,32.194245,35.431655
5,munich,542,37.453875,19.926199,42.619926
0,__root__,206,48.543689,5.825243,45.631068



-------------------- Class distribution per dataset × eyetracker --------------------


...


-------------------- Eyetracker coverage per dataset --------------------


,n_total,n_with_et,pct_with_et
dataset,,,
berlin,7549,999,13.233541
sequences,3483,496,14.240597
barcelona,1198,0,0.000000
london_uk_collideoscope,570,0,0.000000
london_uk_gov,569,0,0.000000
paris,556,0,0.000000
munich,542,0,0.000000
__root__,206,0,0.000000



-------------------- Comparisons per user --------------------


count    285.000000
mean      51.484211
std       35.483242
min        1.000000
10%        5.000000
25%       19.000000
50%       65.000000
75%       65.000000
90%       69.000000
max      333.000000
dtype: float64


Done.
